In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt
import numpy as np

**Skipgram 16/10/2026**

In [1]:
import numpy as np

# 1. Corpus
corpus = ["cats like milk",
          "dogs like balls",
          "cats drink milk",
          "dogs chase cats"]

In [2]:
# 2. Build vocabulary
sentences = [s.split() for s in corpus]
vocab = sorted(set(word for s in sentences for word in s))

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

V = len(vocab)
print("Vocabulary:", vocab)

# Hyperparameters
window = 1
d = 3
learning_rate = 0.05
epochs = 500


Vocabulary: ['balls', 'cats', 'chase', 'dogs', 'drink', 'like', 'milk']


In [3]:
# 3. Generate center-context training pairs
pairs = []

for sentence in sentences:
    ids = [word_to_id[w] for w in sentence]

    for i, center in enumerate(ids):
        for j in range(max(0, i - window),
                       min(len(ids), i + window + 1)):
            if i != j:
                pairs.append((center, ids[j]))

print("Training pairs:", pairs)

Training pairs: [(1, 5), (5, 1), (5, 6), (6, 5), (3, 5), (5, 3), (5, 0), (0, 5), (1, 4), (4, 1), (4, 6), (6, 4), (3, 2), (2, 3), (2, 1), (1, 2)]


In [4]:
# 4. Initialize E (input embeddings) and W (output weights)
np.random.seed(0)
E = np.random.randn(V, d) * 0.1    # (V, d): embedding lookup table
W = np.random.randn(d, V) * 0.1    # (d, V): output weights


In [10]:
# 5. Helper: softmax
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

In [12]:
# 6. Training loop
for epoch in range(epochs):
    np.random.shuffle(pairs)
    total_loss = 0

    for center, context in pairs:
        # Forward pass
        h = E[center]                      # embedding lookup
        scores = h @ W                     # (V,)
        probs = softmax(scores)            # (V,)

        # Loss
        loss = -np.log(probs[context])
        total_loss += loss

        # Backpropagation
        error = probs.copy()
        error[context] -= 1                # probs - one_hot(context)

        grad_W = np.outer(h, error)        # (d, V)
        grad_h = W @ error                 # (d,)  uses old W

        # Update parameters
        W -= learning_rate * grad_W
        E[center] -= learning_rate * grad_h

    if epoch % 100 == 0:
        print(f"epoch {epoch}, loss {total_loss / len(pairs):.4f}")

epoch 0, loss 0.9824
epoch 100, loss 0.9814
epoch 200, loss 0.9804
epoch 300, loss 0.9804
epoch 400, loss 0.9795


In [13]:
# 7. Return E: the learned word vectors
print("\nLearned embeddings:")
for i in range(V):
    print(f"{id_to_word[i]:>6}: {np.round(E[i], 3)}")


Learned embeddings:
 balls: [ 2.031 -3.086 -0.782]
  cats: [-0.828 -0.78  -0.101]
 chase: [2.078 0.12  0.901]
  dogs: [ 0.387 -1.613  1.143]
 drink: [-0.574  1.067  1.913]
  like: [ 0.258  0.951 -0.189]
  milk: [ 0.214 -0.994 -1.733]


In [14]:
# 8. Check the result: most similar words (cosine similarity)
def most_similar(word, top_k=3):
    v = E[word_to_id[word]]
    sims = E @ v / (np.linalg.norm(E, axis=1) * np.linalg.norm(v))
    best = np.argsort(sims)[::-1][1:top_k + 1]   # skip the word itself
    return [(id_to_word[i], round(float(sims[i]), 3)) for i in best]

print("\nSimilar to 'cats':", most_similar("cats"))
print("Similar to 'dogs':", most_similar("dogs"))


Similar to 'cats': [('dogs', 0.357), ('milk', 0.337), ('balls', 0.186)]
Similar to 'dogs': [('balls', 0.64), ('chase', 0.359), ('cats', 0.357)]


In [15]:
# ---- check 1: shortcut loss == full cross-entropy formula ----
center, context = pairs[0]
h = E[center]
probs = softmax(h @ W)

y = np.zeros(V)
y[context] = 1                                  # one-hot target

loss_shortcut = -np.log(probs[context])         # what the training loop uses
loss_full = -np.sum(y * np.log(probs))          # -sum(y_k * log(p_k))
print(loss_shortcut, loss_full, np.isclose(loss_shortcut, loss_full))   # True

# ---- check 2: gradient (probs - y) matches a numerical gradient ----
def loss_fn(scores):
    return -np.log(softmax(scores)[context])

scores = h @ W
analytic = probs - y                            # same as error[context] -= 1

eps = 1e-6
numeric = np.zeros(V)
for k in range(V):
    s_plus, s_minus = scores.copy(), scores.copy()
    s_plus[k] += eps
    s_minus[k] -= eps
    numeric[k] = (loss_fn(s_plus) - loss_fn(s_minus)) / (2 * eps)

print("max gradient difference:", np.abs(analytic - numeric).max())   # ~1e-10

# ---- check 3: loss at initialization should be about log(V) ----
print("loss before training:", loss_shortcut)
print("log(V):", np.log(V))                     # ~1.946 for V = 7

0.007620485866333867 0.007620485866333867 True
max gradient difference: 7.546178005385373e-11
loss before training: 0.007620485866333867
log(V): 1.9459101490553132
